In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

dataset_root = '/kaggle/input/datasets/emmarex/plantdisease/PlantVillage'

print("All classes in the dataset:")
for folder in sorted(os.listdir(dataset_root)):
    print(" -", folder)

In [ ]:
import os
import random
import shutil
from pathlib import Path


dataset_root = '/kaggle/input/datasets/emmarex/plantdisease/PlantVillage'
TARGET_CLASSES = ["Tomato_healthy", "Tomato_Early_blight"]


OUTPUT_ROOT = Path('/kaggle/working/tomato_dataset')
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}
SEED = 42


for split in SPLIT_RATIOS:
    for cls in TARGET_CLASSES:
        (OUTPUT_ROOT / split / cls).mkdir(parents=True, exist_ok=True)


for cls in TARGET_CLASSES:
    src_folder = os.path.join(dataset_root, cls)
    images = [f for f in os.listdir(src_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.Random(SEED).shuffle(images)

    n = len(images)
    n_train = int(n * SPLIT_RATIOS["train"])
    n_val = int(n * SPLIT_RATIOS["val"])

    split_files = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train + n_val:],
    }

    for split, files in split_files.items():
        for fname in files:
            src = os.path.join(src_folder, fname)
            dst = OUTPUT_ROOT / split / cls / fname
            shutil.copy(src, dst)

    print(f"{cls}: total={n} -> train={len(split_files['train'])}, val={len(split_files['val'])}, test={len(split_files['test'])}")

print("\nDone. Dataset saved to:", OUTPUT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, cls in enumerate(TARGET_CLASSES):
    sample_dir = OUTPUT_ROOT / "train" / cls
    sample_files = list(sample_dir.glob('*'))[:4]
    for col, img_path in enumerate(sample_files):
        img = Image.open(img_path)
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls, fontsize=10)
        axes[row, col].axis('off')

plt.suptitle('Sample Images — Train Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/tomato_dataset_co17', 'zip', OUTPUT_ROOT)
print("Zipped and ready at: /kaggle/working/tomato_dataset_co17.zip")